# ¿Podemos exprimir mejor la historia corta?
## Cuatro pruebas sobre el baseline de la cátedra

Una guía para conversar en tutoría: qué cambiamos, cuánto se movió la AUC y qué podemos concluir.

**Antes de mirar los números:** son experimentos exploratorios en agosto de 2019. Ese mes ya se usó para optimización. No es un test independiente. Las semillas miden variación del entrenamiento, no diez meses ni diez muestras nuevas.

Código: rama `feat/historia-transformaciones`. El hash exacto del paquete desplegado y los hashes de cada script quedan en los manifiestos de la corrida.


### La comparación, en una tabla

| Se mantiene igual | Definición |
|---|---|
| Entrenamiento | Marzo-junio de 2019, todas las filas |
| Etiqueta de entrenamiento | BAJA+1 o BAJA+2 |
| Evaluación | AUC de agosto, positivo BAJA+2 |
| Baseline | 152 originales + lag1/2 y delta1/2 = 760 |
| Ajuste | Parámetros del notebook 07, 339 árboles, 10 semillas, 8 hilos |
| Saneamiento | Mismo catálogo y orden de aplicación |

Solo cambia la familia de features añadida. Los parámetros fueron elegidos para el baseline: esta comparación con ajuste fijo no compara los óptimos individuales.

| Familia | Idea | Features nuevas |
|---|---|---:|
| Historia reciente | Media con pesos 4:2:1; diferencia y cambio relativo frente al pasado ponderado 2:1 | 387 |
| Cambios relativos | (actual - previo) / (abs(actual) + abs(previo)), para lag1/2 | 258 |
| Persistencia | Dos caídas, dos subidas, reversión y aceleración relativa | 516 |
| Rankings con cero fijo | Percentiles por mes/signo, luego ranking actual, lags y deltas | 365 |

Faltantes quedan faltantes. Si un operador necesita tres fotos y no están disponibles, no se interpreta el hueco como cero.


### 1. Cargar las salidas

En Colab: ejecutar la próxima celda y subir `tutoria_resultados.zip`, descargado del bucket. En Jupyter local: ajustar `ROOT` a la carpeta descomprimida. Se puede abrir un paquete parcial: la tabla indicará qué terminó y qué falta.

El ZIP contiene métricas y metadatos agregados, sin IDs ni predicciones individuales.


In [ ]:
from pathlib import Path
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path("tutoria_resultados")
try:
    from google.colab import files
except ImportError:
    files = None
if files is not None:
    uploaded = files.upload()
    archives = [name for name in uploaded if name.endswith('.zip')]
    if len(archives) != 1:
        raise ValueError("Subí exactamente un ZIP de resultados")
    ROOT.mkdir(exist_ok=True)
    with zipfile.ZipFile(archives[0]) as z:
        target = ROOT.resolve()
        for member in z.infolist():
            if not (target / member.filename).resolve().is_relative_to(target):
                raise ValueError("Ruta inválida en ZIP")
        z.extractall(ROOT)
print("Resultados:", ROOT.resolve())


In [ ]:
FAMILIES = ["historia_reciente", "cambios_relativos", "persistencia", "rankings_cero_fijo"]
LABELS = {"historia_reciente": "Historia reciente", "cambios_relativos": "Cambios relativos",
          "persistencia": "Persistencia", "rankings_cero_fijo": "Rankings con cero fijo"}
loaded, manifest, states = {}, {}, []
for family in FAMILIES:
    folder = ROOT / family
    status = json.loads((folder / "status.json").read_text()) if (folder / "status.json").exists() else {}
    complete = status.get("status") == "complete"
    states.append({"familia": LABELS[family], "estado": "completo" if complete else "pendiente / incompleto"})
    if not complete:
        continue
    config = json.loads((folder / "config.json").read_text())
    table = pd.read_csv(folder / "metrics.csv")
    if table.duplicated(["variant", "seed"]).any():
        raise ValueError(f"Filas duplicadas en {family}")
    pivot = table.pivot(index="seed", columns="variant", values="auc")
    expected = set(config['seeds_used'])
    if set(pivot.index) != expected or len(expected) != 10 or pivot.isna().any().any():
        raise ValueError(f"Se esperan diez pares completos en {family}")
    if set(pivot.columns) != {'baseline', family}:
        raise ValueError(f"Variantes inesperadas en {family}")
    if not ((pivot >= 0) & (pivot <= 1)).all().all():
        raise ValueError("AUC fuera de rango")
    if config['train'] != [201903,201904,201905,201906] or config['validation'] != 201908:
        raise ValueError("Contrato temporal distinto")
    loaded[family] = pivot
    manifest[family] = config

display(pd.DataFrame(states))
if not loaded:
    raise RuntimeError("Todavía no hay experimentos completos en este paquete. Volvé a descargarlo cuando termine alguno.")

# Los controles deben coincidir entre familias; si no, detener la interpretación.
reference = next(iter(loaded.values()))['baseline']
for family, pivot in loaded.items():
    np.testing.assert_allclose(pivot['baseline'].sort_index(), reference.sort_index(), rtol=0, atol=1e-10,
                               err_msg=f"Baseline diferente en {family}")
print("Controles de integridad OK; familias completas:", len(loaded))


### 2. ¿Cuánto cambió la AUC?

`Δ AUC` está en unidades de AUC: +0,001 significa pasar, por ejemplo, de 0,912 a 0,913. La media de AUC por semilla y la AUC del ensamble son medidas diferentes. El ensamble promedia probabilidades antes de calcular AUC.


In [ ]:
rows, differences = [], {}
for family, pivot in loaded.items():
    d = pivot[family] - pivot['baseline']
    differences[family] = d
    summary = json.loads((ROOT / family / 'summary.json').read_text())
    np.testing.assert_allclose(d.mean(), summary[f'{family}-baseline']['mean_delta'], atol=1e-12)
    rows.append({'familia': LABELS[family], 'AUC baseline media': pivot.baseline.mean(),
                 'AUC variante media': pivot[family].mean(), 'Δ AUC media': d.mean(),
                 'semillas a favor': int((d>0).sum()), 'Δ mínimo': d.min(), 'Δ máximo': d.max(),
                 'AUC ensamble baseline': summary['ensemble_auc']['baseline'],
                 'AUC ensamble variante': summary['ensemble_auc'][family],
                 'Δ ensamble': summary['ensemble_auc'][family] - summary['ensemble_auc']['baseline']})
results = pd.DataFrame(rows).set_index('familia')
display(results.style.format(precision=6))
results.to_csv(ROOT / 'resumen_tutoria.csv')


### 3. ¿La señal se repite entre semillas?

Cada punto es una diferencia variante - baseline para una semilla. La cruz marca la media. Esto muestra dispersión del algoritmo, sin presentarla como un intervalo de generalización temporal.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
for i, (family, d) in enumerate(differences.items()):
    offsets = np.linspace(-.12, .12, len(d))
    ax.scatter(d.to_numpy(), i + offsets, s=35, alpha=.75)
    ax.scatter([d.mean()], [i], marker='x', color='black', s=100, linewidths=2)
ax.axvline(0, color='gray', linestyle='--')
ax.set_yticks(range(len(differences)), [LABELS[f] for f in differences])
ax.set_xlabel('Δ AUC de agosto (variante - baseline)')
ax.set_title('Diez semillas sobre el mismo mes de validación')
ax.grid(axis='x', alpha=.2)
fig.tight_layout()
plt.show()
fig.savefig(ROOT / 'diferencias_auc.png', dpi=180, bbox_inches='tight')

display(pd.DataFrame({LABELS[f]: d for f,d in differences.items()}).rename_axis('semilla').style.format(precision=6))


### 4. ¿Cuánta historia pudimos calcular?

Una mejora podría concentrarse en clientes con historia completa. Esta tabla muestra faltantes de las features nuevas en agosto, sin eliminar clientes de la comparación.


In [ ]:
coverage = []
for family in loaded:
    table = pd.read_csv(ROOT / family / 'extra_coverage.csv')
    coverage.append({'familia': LABELS[family], 'features nuevas': len(table),
                     'faltantes promedio': table.missing_fraction.mean(),
                     'máximo faltantes': table.missing_fraction.max()})
display(pd.DataFrame(coverage).set_index('familia').style.format({'faltantes promedio': '{:.1%}', 'máximo faltantes': '{:.1%}'}))


### Para discutir, sin vender humo

- ¿Mejora la media y también el ensamble? ¿O la aparente mejora depende de algunas semillas?
- ¿El tamaño del efecto justifica agregar cientos de columnas y tiempo de entrenamiento?
- Estas cuatro pruebas forman una exploración: elegir la mejor introduce sesgo de selección. No tratamos sus resultados como cuatro confirmaciones independientes.
- ¿Qué dirección seguiríamos, y qué resultado negativo merece conservarse?
- Agosto ya intervino en ajuste y selección; no prometemos el mismo efecto en otro mes. No abrimos octubre en estas corridas.
- Los pesos recientes quedaron fijados antes de correr; si se prueban otros, serán experimentos nuevos.

**Conclusión de tutoría (completar después de mirar todas las familias):**

> Observamos ... en agosto. La señal fue ... entre semillas. Con estos datos podemos afirmar ...; todavía no podemos afirmar ....


### Anexo: trazabilidad

Los archivos de entrada, scripts y parámetros tienen SHA-256. Las rutas son informativas: no hace falta acceso al disco original para leer las tablas. Para repetir el entrenamiento sí hace falta el mismo dataset y catálogo.


In [ ]:
for family, config in manifest.items():
    print("\n", LABELS[family])
    print("Versiones:", config['versions'])
    print("Semillas:", config['seeds_used'])
    print("Huellas:")
    for path, digest in config['hashes'].items():
        print(" ", Path(path).name, digest)
